# Flights Scraper Sky API

In [1]:
import sys
import os

# Check Python version
print(f"Python Version: `{sys.version}`")  # Detailed version info
print(f"Base Python location: `{sys.base_prefix}`")
print(f"Current Environment location: `{os.path.basename(sys.prefix)}`", end='\n\n')

Python Version: `3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]`
Base Python location: `C:\Users\LMT\AppData\Local\Programs\Python\Python312`
Current Environment location: `.venv`



In [2]:
sys.path.append("../src") if "../src" not in sys.path else None

## Import and Configuration

In [ ]:
import pandas as pd

from datetime import date, timedelta
import os
import requests
import json

import flight_scraper, routes

### `FlightScraper` class

In [4]:
scraper = flight_scraper.FlightScraper()

#### Search one way and parse itineraries

In [5]:
tomorrow = (date.today() + timedelta(days=1)).strftime("%Y-%m-%d")

In [6]:
response = scraper.search_one_way(
    "LHR", "SGN", tomorrow,
    max_polls=10,
)

context = response["data"]["context"]
print(f"Status:  {context['status']}")
print(f"Results: {context['totalResults']}")
print(f"Quota:   {scraper.get_quota_status()}")

records = scraper.parse_itineraries(response)
df = pd.DataFrame(records)
print(f"\nParsed {len(df)} itineraries")
df[["price_raw", "origin_code", "dest_code", "stop_count", "duration_minutes", "carrier_names"]]

Status:  complete
Results: 241
Quota:   {'api_remaining': '19943', 'daily_used': 3, 'daily_limit': 500}

Parsed 241 itineraries


,price_raw,origin_code,dest_code,stop_count,duration_minutes,carrier_names
0,534.00,LHR,SGN,1,1040,Air India
1,332.94,LHR,SGN,2,1180,"Etihad Airways, VietJet Air"
2,332.94,LHR,SGN,2,1250,"Etihad Airways, VietJet Air"
3,350.30,LHR,SGN,2,1250,"Etihad Airways, Sun PhuQuoc Airways"
4,614.79,LHR,SGN,1,955,Qatar Airways
...,...,...,...,...,...,...
236,4112.59,LHR,SGN,1,2230,ANA (All Nippon Airways)
237,3978.22,LHR,SGN,2,2350,Air India
238,4086.35,LHR,SGN,2,2350,"Air India, Singapore Airlines"
239,6832.13,LHR,SGN,1,2160,"Qantas, Vietnam Airlines"


#### Retrieve the price calendar for a specific route starting from the given `depart_date`

In [7]:
cal = scraper.price_calendar("LHR", "SGN", tomorrow)
cal_records = scraper.parse_price_calendar(cal, "LHR", "SGN")
cal_df = pd.DataFrame(cal_records)
cal_df

,collected_at,origin,destination,currency,departure_date,price,group
0,2026-04-01T23:25:22.001557,LHR,SGN,GBP,2026-04-02,490.44,high
1,2026-04-01T23:25:22.001557,LHR,SGN,GBP,2026-04-03,514.00,high
2,2026-04-01T23:25:22.001557,LHR,SGN,GBP,2026-04-04,319.31,low
3,2026-04-01T23:25:22.001557,LHR,SGN,GBP,2026-04-05,479.45,high
4,2026-04-01T23:25:22.001557,LHR,SGN,GBP,2026-04-06,326.13,low
...,...,...,...,...,...,...,...
346,2026-04-01T23:25:22.001557,LHR,SGN,GBP,2027-03-23,606.82,high
347,2026-04-01T23:25:22.001557,LHR,SGN,GBP,2027-03-24,343.77,medium
348,2026-04-01T23:25:22.001557,LHR,SGN,GBP,2027-03-25,564.76,high
349,2026-04-01T23:25:22.001557,LHR,SGN,GBP,2027-03-26,698.16,high


In [8]:
print(f"\nMonthly average prices:")
cal_df["month"] = pd.to_datetime(cal_df["departure_date"]).dt.to_period("M")
cal_df.groupby("month")["price"].agg(["mean", "min", "max"])


Monthly average prices:


,mean,min,max
month,,,
2026-04,358.441034,282.44,514.00
2026-05,345.800323,283.31,449.01
2026-06,388.749333,276.00,560.99
2026-07,564.935484,467.97,730.00
2026-08,471.932333,380.00,712.99
2026-09,359.255333,299.66,471.81
2026-10,410.482333,326.28,509.74
2026-11,365.972333,334.16,464.98
2026-12,483.218710,393.13,652.57


#### Define target routes

In [ ]:

for route in routes.TARGET_ROUTES:
    print(f"  {route['origin']} -> {route['destination']}  "
          f"({route['distance_miles']} mi)  {route['notes'][:50]}")

print(f"\nDaily API budget: {len(routes.TARGET_ROUTES)} calendars + "
      f"{len(routes.TARGET_ROUTES)} searches × 4 polls = "
      f"{len(routes.TARGET_ROUTES) * 5} requests")

  LHR -> SGN  (5980 mi)  Primary target. VN direct via HAN, plus Emirates/S
  LHR -> HAN  (5630 mi)  VN Airlines direct. Also via PEK, CAN, HKG, BKK
  LHR -> BKK  (5930 mi)  BA/TG/EVA direct. Common connection point for SGN/
  SGN -> LHR  (5980 mi)  Return leg. Pricing often asymmetric
  HAN -> LHR  (5630 mi)  Return leg
  BKK -> LHR  (5930 mi)  Return leg

Daily API budget: 6 calendars + 6 searches × 4 polls = 30 requests
